#BERTopic on UNTM Sample Dataset
In this notebook I have implemented Topic Modeling on Urdu News Data transformer based topic modelling technique BERTopic.

## Mounting Google Drive
If the dataset is on Google Drive then you have to mount over google drive with collaboratory.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive




#Installing required dependencies
**One thing to remember is that after installing libraries you have to restart the run time again so that other dependencies are not affected by it.**

In [ ]:
!pip install bertopic

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.1/154.1 kB 1.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 11.1 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.9/90.9 kB 9.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.8/132.8 kB 13.6 MB/s eta 0:00:00
  Using cached Cython-0.29.37-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.manylinux_2_24_x86_64.whl (1.9 MB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.8/55.8 kB 6.5 MB/s eta 0:00:00
  Created wheel for hdbscan: filename=hdbscan-0.8.33-cp310-cp310-linux_x86_64.whl size=3039290 sha256=220acaad32e2b77fab4733bd6a04a153821367e51899f6bc8196397eccf343ca
  Stored in directory: /root/.cache/pip/wheels/75/0b/3b/dc4f60b7cc455efaefb62883a7483e76f09d06ca81cf87d610
  Created wheel for umap-l

In [ ]:
!pip install -U sentence-transformers

In [ ]:
!pip install --upgrade tensorflow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 475.2/475.2 MB 1.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 21.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 442.0/442.0 kB 30.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 41.5 MB/s eta 0:00:00
  Attempting uninstall: tensorflow-estimator
    Found existing installation: tensorflow-estimator 2.12.0
    Uninstalling tensorflow-estimator-2.12.0:
      Successfully uninstalled tensorflow-estimator-2.12.0
  Attempting uninstall: keras
    Found existing installation: keras 2.12.0
    Uninstalling keras-2.12.0:
      Successfully uninstalled keras-2.12.0
  Attempting uninstall: google-auth-oauthlib
    Found existing installation: google-auth-oauthlib 0.4.6
    Uninstalling google-auth-oauthlib-0.4.6:
      Successfully uninstalled google-auth-oauthlib-0.4.6
  Attempting uninstall: tensorboard
    Found existing installation: tensorboard 2.12.0
    Uninstalling te


# Importing required dependencies
We will import numpy, pandas and re, bertopic, gensim library for now. other libraries will be imported in the notebook later.

Pandas will be used to create a Dataframe and handle the csv file. Numpy will be used for the faster computation of arrays to save time. re library will be used for the cleaning of data. gensim library is used to get coherence score. bertopic is used to train bertopic on our UNTM dataset with using pretrained language models

In [ ]:
import pandas as pd
import numpy as np
import re
from bertopic import BERTopic
from gensim.models.coherencemodel import CoherenceModel
import gensim.corpora as corpora
#optional
import logging
logging.basicConfig(format='%(asctime)s : %(levelname)s : %(message)s', level=logging.ERROR)

import warnings
warnings.filterwarnings("ignore",category=DeprecationWarning)

##DataFrame




In [ ]:
#load UNTM Dataset
df = pd.read_csv('/content/drive/MyDrive/sampled_UNTM.csv', encoding='utf-8')

In [ ]:
print(len(df))
df.head()

399


,Date,Title,News,Category,Link,Clean_Data
0,2023-03-02,چیئرمین آئی پی او کا کاپی رائٹس سٹیک ہولڈرز کے...,لاہور (اُردو پوائنٹ اخبارتازہ ترین - اے پی پی۔...,Business,https://www.urdupoint.com/business/news-detail...,انٹلیکچوئل پراپرٹی رائٹس آرگنائزیشن کے چیئرم...
1,2023-03-17,سٹیٹ بینک پارلیمنٹ کے ایوان بالا کی گولڈن جوبل...,اسلام آباد (اُردو پوائنٹ اخبارتازہ ترین - اے پ...,Business,https://www.urdupoint.com/business/news-detail...,سٹیٹ بینک پارلیمنٹ کے ایوان بالا سینیٹ کی گول...
2,2023-04-19,دیر چیمبر آف کامرس اینڈ انڈسٹری کی تعاون سے دی...,دیر لو ئر، 19 اپریل (اُردو پوائنٹ اخبارتازہ تر...,Business,https://www.urdupoint.com/business/news-detail...,ڈپٹی کمشنر کمانڈنٹ دیر لیویز افتخار احمد اور ...
3,2023-04-05,سیالکوٹ چیمبر آف کامرس کا شیخ ریاض الدین اور خ...,سیالکوٹ (اُردو پوائنٹ اخبارتازہ ترین - اے پی پ...,Business,https://www.urdupoint.com/business/news-detail...,سیالکوٹ چیمبر آف کامرس کے سینئر نائب صدر وہب ...
4,2023-04-21,برازیل میں ہائبرڈ گاڑیوں کی تیاری کے لیے 34 کر...,ٹوکیو (اُردو پوائنٹ اخبارتازہ ترین - اے پی پی۔...,Business,https://www.urdupoint.com/business/news-detail...,جاپان کی کار ساز کمپنی ٹویوٹا نے کہا ہے کہ وہ...


## Preprocessing of Data
Data was cleaned that saved into column "Clean_Data" we  removed some extra coloumns first like date,Title,Link and some other columns too. Because we do experiment only news  and all other things are extra for us.

Stopwords are common words that are often filtered out during text processing in natural language processing (NLP) tasks. These words are considered to have little or no value in conveying the actual meaning of the text. We take list of 401 stopwords for topic modelling.

In [ ]:
df=df.drop(columns=['Date','Title', 'Link'])

In [ ]:
def remove_nonbreaking_space(text):
    return re.sub(r'\xa0', ' ', text)

df['Clean_Data'] = df['Clean_Data'].apply(remove_nonbreaking_space)

In [ ]:
documents = df['Clean_Data'].values.tolist()
print((documents[1]))

 سٹیٹ بینک پارلیمنٹ کے ایوان بالا سینیٹ کی گولڈن جوبلی کے موقع پر  روپے کا یادگاری سکہ جاری کرے گا۔سٹیٹ بینک کی جانب سے جاری اعلامیہ کے مطابق سینیٹ آف پاکستان جو کہ آئینی طور پر ہاؤس آف فیڈریشن ہے پاکستان کی دو ایوانوں پر مشتمل پارلیمنٹ کی قانون سازی کا بالائی چیمبر ہے کی تشکیل کی  سال مکمل ہونے پر  روپے کا یادگاری سکہ جاری کیا جائے گا ۔ اعلامیہ میں کہا گیا ہے کہ سینٹ آف پاکستان میں پاکستان کے تمام صوبوں کو مساویانہ نمائندگی دی گئی ہے یہ ایک مستقل ایوان ہے جو قومی امور کے پروسیس میں تسلسل کی علامت ہے۔ سال ء پاکستان کے سینٹ کی گولڈن جوبلی کا سال ہے۔ اس خصوصی موقع کی مناسبت سے وفاقی حکومت نے سٹیٹ بینک کو  روپے کا یادگاری سکہ جاری کرنے کی اجازت دی ہے۔     یہ یادگاری سکہ  مارچ ء سے ایس بی پی بینکنگ سروسز کارپوریشن کے فیلڈ دفاتر کے تمام تبادلہ کاؤنٹرز سے جاری کیا جائے گا۔ روپے کے یادگاری سکے کی شکل گول ہے جسے  ملی میٹر سمت کے ساتھ نقش کیا گیا ہے اس کا وزن  گرام اور یہ تانبے اور نکل کے دھاتی اجزا  پرمشتمل ہے۔ سکے کے سامنے کے رخ کے درمیان میں بڑھتا ہوا ہلالی چاند اور پانچ نوکوں پر مشتمل ستار

In [ ]:
import nltk
stopwords=pd.read_csv('/content/drive/MyDrive/stopwords.txt',names=['List'])
# stopwords['List']

stopwords_ur=[]
for li in stopwords['List']:
  stopwords_ur.append(li)
print(len(stopwords_ur))
print(stopwords_ur)

401
['کی', 'ہیں', 'ہے', 'رہا', 'رہی', 'رہے', 'تھا', 'تھے', 'تھی', 'تھى', 'میں', 'کہ', 'کے', 'ہوتے', 'کہہ', 'بنانا', 'پھر', 'لکین', 'ہوتی', 'لیتی', 'ایسی', 'ائے', 'ہوئے', 'ہوۓ ', 'مگر', 'چاہے', 'کیے', 'تاکہ', 'تم', 'آکر', 'لگا', 'ہوگیئں', 'ليے', 'رہتی', 'ہوگئی', 'انھوں', 'چاہتے', 'پاگیئں', 'آنا', 'کروا', 'رکھ', 'آتی', 'یہاں', 'جیسا', 'چکے', 'کرئے', 'دیے', 'چکا', 'ملتا', 'کبھی', 'ایسا', 'کرسکیں', 'ہوسکے', 'سکیں', 'لہذا', 'چاہئے', 'وہیں', 'اٹھایا', 'جہنوں', 'ہمارے', 'لیے', 'آرہے', 'کرگیئں', 'بنانے', 'سکتا', 'وغیرہ', 'دے', 'ہوۓ', 'رہنے', 'ہوۓ', 'کئے', 'لگے', 'لگایا', 'لائے', 'کہے', 'کرے', 'رہئں', 'آگئے', 'کئی', 'کم', 'ملی', 'جنہیں', 'ہوئیں', 'تھیں', 'ابھی', 'پاگئیں', 'آئے', 'کرا', 'دیا', 'جہاں', 'آگئیں', 'کرتی', 'رہیں', 'کرتیں', 'دیتی', 'ہوگئے', 'ديتے', 'انہں', 'ایسے', 'رکھتے', 'رہتے', 'رکھی', 'کیں', 'كیا', 'وه', 'جنہيں', 'کروائی', 'دینا', 'جسے', 'جاتی', 'اسکی', 'ملے', 'کرگئى', 'جن', 'آپکو', 'جنہوں', 'دیئے', 'والی', 'جبکہ', 'دیگر', 'آپکا', 'رکھنے', 'انہىں', 'کيے', 'یعنی', 'كيے', 'بننے', 'ج

# BERTopic Training


In [ ]:
#create custom embedding
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2')
embeddings = model.encode(documents, show_progress_bar=True)
print(embeddings)

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/4.09k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

1_Pooling/config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/13 [00:00<?, ?it/s]

[[-0.14562157  0.12676856 -0.25911704 ... -0.08738136 -0.13059202
   0.23454961]
 [ 0.04244933  0.06325082  0.15818478 ...  0.18492761  0.09699746
   0.00209925]
 [-0.03406477  0.27974227  0.23134182 ... -0.04284064  0.0348863
   0.25003275]
 ...
 [-0.12654519 -0.01108428 -0.15280664 ...  0.11761579  0.17626058
  -0.04933152]
 [-0.00931564 -0.09762228  0.23743257 ... -0.0712635   0.12250552
   0.09448026]
 [ 0.03949545  0.10941717  0.04547139 ...  0.04623723 -0.0348675
   0.18558027]]


In [ ]:
#pass vectorizer_model to bertopic for stopwords removal
from sklearn.feature_extraction.text import CountVectorizer

vectorizer_model = CountVectorizer(stop_words= stopwords_ur)


In [ ]:
#ClassTfidf for top words
from bertopic.vectorizers import ClassTfidfTransformer

ctfidf_model = ClassTfidfTransformer(bm25_weighting=True, reduce_frequent_words=True)

In [ ]:
#UMAP for dimention reduction
from umap import UMAP
dim_model = UMAP(n_neighbors=15, n_components=2, random_state=42)

In [ ]:
# # #KMeans used for clustering
from sklearn.cluster import KMeans

cluster_model = KMeans(n_clusters=7, random_state=42)

In [ ]:
# from sklearn.cluster import AgglomerativeClustering

# cluster_model = AgglomerativeClustering(n_clusters=7)

In [ ]:
np.random.seed(42)

In [ ]:
topic_model = BERTopic(language="urdu", low_memory=True ,calculate_probabilities=True, vectorizer_model=vectorizer_model, top_n_words=10,hdbscan_model=cluster_model, umap_model=dim_model,ctfidf_model=ctfidf_model, verbose=True)

In [ ]:
#Fit documents in bertopic
topics, probs = topic_model.fit_transform(documents,embeddings)

2024-02-04 06:06:07,622 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2024-02-04 06:06:24,553 - BERTopic - Dimensionality - Completed ✓
2024-02-04 06:06:24,555 - BERTopic - Cluster - Start clustering the reduced embeddings
2024-02-04 06:06:24,655 - BERTopic - Cluster - Completed ✓
2024-02-04 06:06:24,668 - BERTopic - Representation - Extracting topics from clusters using representation models.
2024-02-04 06:06:24,907 - BERTopic - Representation - Completed ✓


In [ ]:
print(probs)

None


In [ ]:
#topics that assign to each document
print(topics)

[2, 1, 2, 1, 3, 3, 1, 3, 3, 2, 2, 2, 3, 2, 1, 3, 3, 1, 3, 3, 3, 3, 3, 3, 3, 0, 3, 2, 2, 3, 2, 3, 3, 2, 3, 3, 2, 2, 3, 2, 3, 3, 2, 2, 3, 2, 3, 2, 3, 3, 3, 3, 1, 1, 2, 3, 3, 3, 2, 3, 6, 6, 2, 2, 6, 1, 6, 6, 6, 6, 2, 2, 3, 6, 6, 6, 6, 6, 6, 6, 2, 6, 6, 1, 1, 6, 2, 3, 6, 6, 6, 6, 6, 6, 2, 6, 6, 2, 1, 2, 3, 2, 2, 2, 2, 6, 2, 6, 2, 2, 6, 2, 2, 6, 6, 6, 6, 1, 6, 3, 0, 1, 1, 1, 1, 4, 1, 5, 1, 3, 1, 1, 1, 1, 1, 2, 1, 0, 1, 1, 1, 1, 4, 1, 1, 1, 1, 1, 1, 1, 1, 3, 1, 1, 4, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 5, 1, 4, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 4, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 6, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 5, 5, 2, 5, 5, 2, 5, 2, 0, 5, 2, 5, 5, 5, 0, 5, 5, 5, 2, 1, 5, 2, 5, 0, 5, 1, 5, 5, 5, 5, 5, 1, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 1, 4, 3, 4, 4, 3, 2, 2, 4, 0, 4, 4, 4, 2, 6, 4, 4, 2, 4, 3, 2, 2, 2, 4, 4, 4, 3, 4, 0, 4, 2, 3, 6, 3, 4, 3, 4, 2, 2, 0, 3, 6, 4, 4, 4, 4, 2, 2, 4, 2, 

In [ ]:
topic_model.get_topic_freq()

,Topic,Count
3,0,78
1,1,63
0,2,62
2,3,57
5,4,54
6,5,45
4,6,40


In [ ]:
document_topics = topic_model.get_topics()

In [ ]:
#topics with score
print(document_topics)

{0: [('فلم', 0.44275789558846906), ('اداکارہ', 0.38408422310038665), ('اداکار', 0.36712812193421746), ('ویڈیو', 0.3352164085815292), ('میڈیا', 0.31514861236356934), ('بھارتی', 0.3139812173173525), ('شادی', 0.30639769915328513), ('بالی', 0.3044453528967237), ('بیٹے', 0.3043840025651749), ('خاتون', 0.3018150768631854)], 1: [('عمران', 0.40261386887901424), ('خان', 0.39929043941504994), ('حکومت', 0.37370932935342094), ('الیکشن', 0.3567845509945814), ('وزیراعظم', 0.351768415430834), ('شریف', 0.34605147160805516), ('لیگ', 0.3403391620121665), ('تحریک', 0.3346533186376195), ('نواز', 0.3231909597489307), ('کوئی', 0.3094353924381431)], 2: [('پی', 0.3325399968829313), ('ٹی', 0.32982045362943463), ('سی', 0.3235533314582003), ('پاکستان', 0.31785546220971544), ('ڈاکٹر', 0.30404252847725094), ('صحت', 0.2946258697601218), ('چیمبر', 0.28423365693459646), ('پنجاب', 0.27816860660382553), ('کمیٹی', 0.27735762290925636), ('بزنس', 0.2699486100057828)], 3: [('روپے', 0.49436267814746204), ('بک', 0.4229218882

In [ ]:
#shows which documents assign to specific topic
topic_model.get_representative_docs(topic=None)

{0: [' ہدایت کار کامران شاہد کی نئی پاکستانی فلم ہوئے تم اجنبی کے شاندار میوزک کی تعارفی تقریب کا اہتمام کراچی کے مقامی سینماء گھر میں کیا گیا جس میں شوبز کے چمکتے ستاروں سمیت اہم سیاسی شخصیات اور شخصیات نے بڑی تعداد میں شرکت کی۔اس موقع پر باصلاحیت ہدایت کار اور سکرپٹ رائٹر کامران شاہد کے علاوہ اداکار سہیل احمد  علی خان شمعون عباسی محمود اسلم فیصل قریشی جاوید شیخ اور فلم کے ہیرو میکال ذوالفقار بھی موجود تھے۔ میڈیا وال پر موجود فوٹوگرافرز اور کیمرہ مین اپنے کیمروں کے ذریعے ان کی تصاویر اور ویڈیوز بناتے نظر آرہے تھے۔بین الاقوامی شہرت یافتہ گلوکار علی ظفر کا گایا ہوا فلم کا ٹائٹل سونگ ہوئے تم اجنبی کی ویڈیو حاضرین کو دکھائی گئی ۔بین الاقوامی شہرت یافتہ بانسری کے ماہر اور میوزک کمپوزر باقر عباس کے کمپوز کردہ اس گانے کو علی ظفر جیسے شاندار گلوکار نے اپنی سریلی آواز میں پیش کیا۔ٹائٹل گانے کی ویڈیو کے بعد فلم کی منٹ کی جھلکیاں پیش کی گئیں جس میں فلم کے دلچسپ سین کے ساتھ ساتھ دو مزید گانوں کی جھلکیاں بھی شامل تھیں۔ اس رومانوی داستان میں صوفیانہ موسیقی کی ملکہ عابدہ پروین باقر عباس ساحر علی ب

In [ ]:
topic_model.get_topic_info()

,Topic,Count,Name,Representation,Representative_Docs
0,0,78,0_فلم_اداکارہ_اداکار_ویڈیو,"[فلم, اداکارہ, اداکار, ویڈیو, میڈیا, بھارتی, ش...",[ ہدایت کار کامران شاہد کی نئی پاکستانی فلم ہو...
1,1,63,1_عمران_خان_حکومت_الیکشن,"[عمران, خان, حکومت, الیکشن, وزیراعظم, شریف, لی...",[کیا سابق چیف جسٹس گلگت بلتستان نے لندن میں بی...
2,2,62,2_پی_ٹی_سی_پاکستان,"[پی, ٹی, سی, پاکستان, ڈاکٹر, صحت, چیمبر, پنجاب...",[کراچی گوگل کے نیکسٹ بلین یوزرز اقدام کے تحت ...
3,3,57,3_روپے_بک_کمی_مالی,"[روپے, بک, کمی, مالی, فیس, ڈالر, فروخت, فروری,...",[اسلام آباد ریاستہائے متحدہ امریکہ کوویکس کے ...
4,4,54,4_فون_ویوو_اسمارٹ_صارفین,"[فون, ویوو, اسمارٹ, صارفین, سمارٹ, ٹیکنالوجی, ...",[لاہور نے اپنے پہلے اسمارٹ فون کی رونمائی کر د...
5,5,45,5_ٹیم_میچ_ٹورنامنٹ_ہاکی,"[ٹیم, میچ, ٹورنامنٹ, ہاکی, کرکٹ, کپتان, کھلاڑی...",[ نگران وزیراعلی پنجاب سید محسن نقوی کی ہدایت ...
6,6,40,6_کورونا_وائرس_ویکسین_ہیلتھ,"[کورونا, وائرس, ویکسین, ہیلتھ, گھنٹوں, کیسز, ہ...",[ ضلع فیصل آبادکے کوروناویکسی نیشن سنٹرز پراب ...


In [ ]:
topic_model.get_document_info(documents)

,Document,Topic,Name,Representation,Representative_Docs,Top_n_words,Representative_document
0,انٹلیکچوئل پراپرٹی رائٹس آرگنائزیشن کے چیئرم...,2,2_پی_ٹی_سی_پاکستان,"[پی, ٹی, سی, پاکستان, ڈاکٹر, صحت, چیمبر, پنجاب...",[کراچی گوگل کے نیکسٹ بلین یوزرز اقدام کے تحت ...,پی - ٹی - سی - پاکستان - ڈاکٹر - صحت - چیمبر -...,False
1,سٹیٹ بینک پارلیمنٹ کے ایوان بالا سینیٹ کی گول...,1,1_عمران_خان_حکومت_الیکشن,"[عمران, خان, حکومت, الیکشن, وزیراعظم, شریف, لی...",[کیا سابق چیف جسٹس گلگت بلتستان نے لندن میں بی...,عمران - خان - حکومت - الیکشن - وزیراعظم - شریف...,False
2,ڈپٹی کمشنر کمانڈنٹ دیر لیویز افتخار احمد اور ...,2,2_پی_ٹی_سی_پاکستان,"[پی, ٹی, سی, پاکستان, ڈاکٹر, صحت, چیمبر, پنجاب...",[کراچی گوگل کے نیکسٹ بلین یوزرز اقدام کے تحت ...,پی - ٹی - سی - پاکستان - ڈاکٹر - صحت - چیمبر -...,False
3,سیالکوٹ چیمبر آف کامرس کے سینئر نائب صدر وہب ...,1,1_عمران_خان_حکومت_الیکشن,"[عمران, خان, حکومت, الیکشن, وزیراعظم, شریف, لی...",[کیا سابق چیف جسٹس گلگت بلتستان نے لندن میں بی...,عمران - خان - حکومت - الیکشن - وزیراعظم - شریف...,False
4,جاپان کی کار ساز کمپنی ٹویوٹا نے کہا ہے کہ وہ...,3,3_روپے_بک_کمی_مالی,"[روپے, بک, کمی, مالی, فیس, ڈالر, فروخت, فروری,...",[اسلام آباد ریاستہائے متحدہ امریکہ کوویکس کے ...,روپے - بک - کمی - مالی - فیس - ڈالر - فروخت - ...,False
...,...,...,...,...,...,...,...
394,بولیویا کے تین لڑکے اس وقت ہسپتال پہنچ گئے جب...,0,0_فلم_اداکارہ_اداکار_ویڈیو,"[فلم, اداکارہ, اداکار, ویڈیو, میڈیا, بھارتی, ش...",[ ہدایت کار کامران شاہد کی نئی پاکستانی فلم ہو...,فلم - اداکارہ - اداکار - ویڈیو - میڈیا - بھارت...,False
395,ٹوکیو کے شن اوبوکو مضافات میں ایک ایسا انوکھا ...,4,4_فون_ویوو_اسمارٹ_صارفین,"[فون, ویوو, اسمارٹ, صارفین, سمارٹ, ٹیکنالوجی, ...",[لاہور نے اپنے پہلے اسمارٹ فون کی رونمائی کر د...,فون - ویوو - اسمارٹ - صارفین - سمارٹ - ٹیکنالو...,False
396,آسٹریلیا کی ایک پیزا شاپ نے آسٹریلوی فائرفائٹ...,3,3_روپے_بک_کمی_مالی,"[روپے, بک, کمی, مالی, فیس, ڈالر, فروخت, فروری,...",[اسلام آباد ریاستہائے متحدہ امریکہ کوویکس کے ...,روپے - بک - کمی - مالی - فیس - ڈالر - فروخت - ...,False
397,دہلی کے لوک نائیک ہسپتال کے ایک بستر پر ماہ ...,0,0_فلم_اداکارہ_اداکار_ویڈیو,"[فلم, اداکارہ, اداکار, ویڈیو, میڈیا, بھارتی, ش...",[ ہدایت کار کامران شاہد کی نئی پاکستانی فلم ہو...,فلم - اداکارہ - اداکار - ویڈیو - میڈیا - بھارت...,False


In [ ]:
topic_distr, _ = topic_model.approximate_distribution(documents, window=3, min_similarity=0.01)

100%|██████████| 1/1 [00:00<00:00,  1.47it/s]


In [ ]:
print(topic_distr)

[[0.10215937 0.15652699 0.32228055 ... 0.09780149 0.09355867 0.1096797 ]
 [0.1117398  0.27715831 0.14734692 ... 0.10775771 0.11725155 0.09284228]
 [0.11217489 0.17139389 0.308853   ... 0.09787739 0.09151896 0.11869382]
 ...
 [0.10612152 0.09075082 0.12305417 ... 0.12415329 0.11247949 0.10194984]
 [0.34220481 0.096599   0.1200349  ... 0.15068059 0.10749395 0.10213763]
 [0.09168212 0.37417868 0.09674426 ... 0.12780244 0.14242597 0.07910225]]


In [ ]:
topic_model.visualize_distribution(topic_distr[0], width=600,height=600, title="Topic Probability Distributio(UNTM)")

# Evaluation
we used three evaluation metrics to compare the results.

1. The coherence score is used to capture the degree of similarity between the words within each topic, with higher scores indicating more coherent topics. We used two coherence metrics NPMI and Cv Score.
2. IRBO measures are used to assess how different and distinct the topics are in a topic model.


### Coherence Score
To evaluate the model topics coherence we use [Gensim](https://radimrehurek.com/gensim/models/coherencemodel.html) library

In [ ]:
texts = [[word for word in str(document).split() if word not in stopwords_ur] for document in documents]
id2word = corpora.Dictionary(texts)
corpus = [id2word.doc2bow(text) for text in texts]

In [ ]:
topics_bert=[]
for i in topic_model.get_topics():
  row=[]
  topic= topic_model.get_topic(i)
  for word in topic:
     row.append(word[0])
  topics_bert.append(row)

In [ ]:
print(topics_bert)

[['فلم', 'اداکارہ', 'اداکار', 'ویڈیو', 'میڈیا', 'بھارتی', 'شادی', 'بالی', 'بیٹے', 'خاتون'], ['عمران', 'خان', 'حکومت', 'الیکشن', 'وزیراعظم', 'شریف', 'لیگ', 'تحریک', 'نواز', 'کوئی'], ['پی', 'ٹی', 'سی', 'پاکستان', 'ڈاکٹر', 'صحت', 'چیمبر', 'پنجاب', 'کمیٹی', 'بزنس'], ['روپے', 'بک', 'کمی', 'مالی', 'فیس', 'ڈالر', 'فروخت', 'فروری', 'سال', 'ملین'], ['فون', 'ویوو', 'اسمارٹ', 'صارفین', 'سمارٹ', 'ٹیکنالوجی', 'ملازمین', 'کار', 'استعمال', 'ساتھ'], ['ٹیم', 'میچ', 'ٹورنامنٹ', 'ہاکی', 'کرکٹ', 'کپتان', 'کھلاڑیوں', 'کپ', 'بابر', 'سیریز'], ['کورونا', 'وائرس', 'ویکسین', 'ہیلتھ', 'گھنٹوں', 'کیسز', 'ہزار', 'مریض', 'سنٹر', 'ٹیسٹ']]


In [ ]:
# compute Coherence Score CV and NPMI

cm = CoherenceModel(topics=topics_bert, texts=texts, dictionary=id2word, coherence='c_npmi')
coherence = round(cm.get_coherence(),2)
print('\nCoherence Score: ', coherence)


Coherence Score:  0.02


**Diversity Score**

In [ ]:
import itertools
from rbo import rbo
import numpy as np

class InvertedRBO:
    def __init__(self):
        pass

    def irbo(self, topics, topk=10, weight=0.9):
        """
        Calculate inverted Rank Biased Overlap (RBO) as a measure of topic diversity from a list of lists of words.

        :param topics: A list of lists of words representing different topics.
        :param topk: The number of top words on which RBO will be computed.
        :param weight: Weight of each agreement at depth d: p**(d-1). When set to 1.0, there is no weight,
                       and the RBO returns to average overlap.
        :return: The inverted RBO topic diversity score.
        """
        if topk <= 0:
            raise ValueError("topk must be a positive integer.")

        num_topics = len(topics)
        if num_topics == 0:
            raise ValueError("topics list cannot be empty.")

        if topk > len(topics[0]):
            raise Exception('Words in topics are less than topk')

        collect = []
        for list1, list2 in itertools.combinations(topics, 2):
            rbo_val = rbo(list1[:topk], list2[:topk], p=weight)[2]
            collect.append(rbo_val)

        Irbo_score = 1 - np.mean(collect)
        return Irbo_score

In [ ]:
inverted_rbo_calculator = InvertedRBO()
IRBO= round(inverted_rbo_calculator.irbo(topics_bert, topk=10, weight=0.9),2)
print("Inverted RBO Score:", IRBO)

Inverted RBO Score: 1.0


# Visualize Topics

In [ ]:
topic_model.visualize_topics()

In [ ]:
topic_model.visualize_barchart(n_words=10,width=220, height=270, title="Topic Word Scores (UNTM)")

# Model serialization

In [ ]:
# Save model
topic_model.save("my_model")

2024-02-04 06:12:01,263 - BERTopic - WARNING: When you use `pickle` to save/load a BERTopic model,please make sure that the environments in which you saveand load the model are **exactly** the same. The version of BERTopic,its dependencies, and python need to remain the same.


In [ ]:
loaded_topic_model = topic_model.load("my_model")
loaded_topic_model.get_topic_info()

,Topic,Count,Name,Representation,Representative_Docs
0,0,78,0_فلم_اداکارہ_اداکار_ویڈیو,"[فلم, اداکارہ, اداکار, ویڈیو, میڈیا, بھارتی, ش...",[ ہدایت کار کامران شاہد کی نئی پاکستانی فلم ہو...
1,1,63,1_عمران_خان_حکومت_الیکشن,"[عمران, خان, حکومت, الیکشن, وزیراعظم, شریف, لی...",[کیا سابق چیف جسٹس گلگت بلتستان نے لندن میں بی...
2,2,62,2_پی_ٹی_سی_پاکستان,"[پی, ٹی, سی, پاکستان, ڈاکٹر, صحت, چیمبر, پنجاب...",[کراچی گوگل کے نیکسٹ بلین یوزرز اقدام کے تحت ...
3,3,57,3_روپے_بک_کمی_مالی,"[روپے, بک, کمی, مالی, فیس, ڈالر, فروخت, فروری,...",[اسلام آباد ریاستہائے متحدہ امریکہ کوویکس کے ...
4,4,54,4_فون_ویوو_اسمارٹ_صارفین,"[فون, ویوو, اسمارٹ, صارفین, سمارٹ, ٹیکنالوجی, ...",[لاہور نے اپنے پہلے اسمارٹ فون کی رونمائی کر د...
5,5,45,5_ٹیم_میچ_ٹورنامنٹ_ہاکی,"[ٹیم, میچ, ٹورنامنٹ, ہاکی, کرکٹ, کپتان, کھلاڑی...",[ نگران وزیراعلی پنجاب سید محسن نقوی کی ہدایت ...
6,6,40,6_کورونا_وائرس_ویکسین_ہیلتھ,"[کورونا, وائرس, ویکسین, ہیلتھ, گھنٹوں, کیسز, ہ...",[ ضلع فیصل آبادکے کوروناویکسی نیشن سنٹرز پراب ...
